In [ ]:
# Missing Values Detection
import pandas as pd
import json

with open("Charity Filtered.json", "r", encoding="utf-8-sig") as f:
    data = json.load(f)
df = pd.DataFrame(data)
col = "charity_reporting_status"
missing_ratio = df[col].isna().mean()
print(f"{col} missing ratio: {missing_ratio:.2%}")

In [ ]:
# Remove Removed Charities
import pandas as pd
import json

with open("Charity.json", "r", encoding="utf-8-sig") as f:
    data = json.load(f)
df = pd.DataFrame(data)
df_filtered = df[df["charity_registration_status"] != "Removed"]
df_filtered.to_json(
    "Charity Filtered.json",
    orient="records",
    force_ascii=False,
    indent=2
)
print(f"Original rows: {len(df)}")
print(f"Remaining rows: {len(df_filtered)}")

In [ ]:
# Charity+Parta Test Join
import pandas as pd

charity = pd.read_json("Charity Filtered.json", encoding="utf-8-sig")[["organisation_number"]]
charity = charity.drop_duplicates(subset="organisation_number")
parta = pd.read_json("Charity Parta.json", encoding="utf-8-sig")
parta_latest = parta[parta["latest_fin_period_submitted_ind"] == True].copy()
parta_latest["fin_period_end_date"] = pd.to_datetime(parta_latest["fin_period_end_date"], errors="coerce")
parta_latest = parta_latest.sort_values("fin_period_end_date", ascending=False).drop_duplicates(subset="organisation_number")
df = pd.merge(charity, parta_latest, on="organisation_number", how="inner")
df = df.drop_duplicates()
df.to_json("charity_parta_latest_inner_all_columns.json", orient="records", indent=2, force_ascii=False)
print("Finished. Rows: ", len(df))

In [ ]:
# Charity+Area Test Join
import pandas as pd

new_data = pd.read_json("Charity Filtered.json", encoding="utf-8-sig")
area = pd.read_json("Charity Area of Operation.json", encoding="utf-8-sig")
area_grouped = area.groupby("organisation_number").agg({"geographic_area_description": lambda x: list(pd.unique(x)),"geographic_area_type": lambda x: list(pd.unique(x))}).reset_index()

LEVEL_PRIORITY = {
    "International": 4,
    "National": 3,
    "Regional": 2,
    "Local": 1,
    "Unknown": 0
}

def map_service_level(types, descriptions):
    levels = []
    for t, d in zip(types, descriptions):
        if t == "Local Authority":
            levels.append("Local")
        elif t == "Region":
            levels.append("Regional")
        elif t == "Country":
            if d in ["Scotland", "Northern Ireland"]:
                levels.append("National")
            else:
                levels.append("International")
        else:
            levels.append("Unknown")
    if levels:
        return max(levels, key=lambda x: LEVEL_PRIORITY.get(x, 0))
    else:
        return "Unknown"

def compute_service_level(row):
    types = row['geographic_area_type']
    descs = row['geographic_area_description']
    min_len = min(len(types), len(descs))
    return map_service_level(types[:min_len], descs[:min_len])

area_grouped['service_level'] = area_grouped.apply(compute_service_level, axis=1)
df = pd.merge(new_data, area_grouped, on="organisation_number", how="inner")
df = df.drop_duplicates(subset=["organisation_number"])

df.to_json("charity_area_joined.json", orient="records", indent=2, force_ascii=False)
print("Finished. Rows:", len(df))

In [11]:
# Area+Charity+Parta Join Final
import pandas as pd

charity = pd.read_json("Charity Filtered.json", encoding="utf-8-sig")
parta = pd.read_json("Charity Parta.json", encoding="utf-8-sig")
area = pd.read_json("Charity Area of Operation.json", encoding="utf-8-sig")
parta_latest = parta[parta["latest_fin_period_submitted_ind"] == True]
parta_latest = parta[parta["latest_fin_period_submitted_ind"] == True].copy()
parta_latest["fin_period_end_date"] = pd.to_datetime(parta_latest["fin_period_end_date"], errors="coerce")
parta_latest = parta_latest.sort_values("fin_period_end_date", ascending=False).drop_duplicates(subset=["organisation_number"])

area["pair"] = list(zip(area["geographic_area_type"], area["geographic_area_description"]))
area_grouped = area.groupby("organisation_number").agg({"pair": list,"geographic_area_description": lambda x: list(pd.unique(x)), "geographic_area_type": lambda x: list(pd.unique(x))}).reset_index()

LEVEL_PRIORITY = {"International": 4, "National": 3, "Regional": 2, "Local": 1, "Unknown": 0}
def map_service_level(pairs):
    levels = []
    for t, d in pairs:
        d = str(d).strip()
        if t == "Local Authority":
            levels.append("Local")
        elif t == "Region":
            levels.append("Regional")
        elif t == "Country":
            if d in ["Scotland", "Northern Ireland"]:
                levels.append("National")
            else:
                levels.append("International")
        else:
            levels.append("Unknown")
    return max(levels, key=lambda x: LEVEL_PRIORITY.get(x,0)) if levels else "Unknown"

area_grouped['service_level'] = area_grouped["pair"].apply(map_service_level)
area_grouped = area_grouped.drop(columns=["pair"])
df = charity.merge(parta_latest, on="organisation_number", how="inner")
df = df.merge(area_grouped, on="organisation_number", how="inner")
df = df.drop_duplicates(subset=["organisation_number"])
df.to_json("charity_parta_area_joined.json", orient="records", indent=2, force_ascii=False)
print("Finished. Rows:", len(df))

Finished. Rows: 113416


In [ ]:
# Stratify Process
import pandas as pd

charity = pd.read_json("charity_parta_area_joined.json", encoding="utf-8-sig")
need_cols = [
    "total_gross_income",
    "grant_making_is_main_activity",
    "service_level",
]
missing = [c for c in need_cols if c not in charity.columns]
assert not missing, f"Missing Attributes: {missing}"

charity["total_gross_income"] = pd.to_numeric(charity["total_gross_income"], errors="coerce")
merged_df = charity[charity["total_gross_income"] > 0].copy()
merged_df = merged_df.dropna(subset=["grant_making_is_main_activity", "service_level"])
merged_df["income_band"] = pd.qcut(merged_df["total_gross_income"], q=5, labels=["q1","q2","q3","q4","q5"], duplicates="drop")
group_cols = ["income_band", "service_level"]

def stratified_sample(df, per_stratum=15, ratio_0_1=(9,2), random_state=42):
    samples = []
    for _, group in df.groupby(group_cols, observed=True):
        grp_0 = group[group["grant_making_is_main_activity"] == 0]
        grp_1 = group[group["grant_making_is_main_activity"] == 1]
        total_needed = min(per_stratum, len(group))
        n0 = int(total_needed * ratio_0_1[0] / sum(ratio_0_1))
        n1 = total_needed - n0
        n0 = min(n0, len(grp_0))
        n1 = min(n1, len(grp_1))
        if n0 > 0:
            samples.append(grp_0.sample(n=n0, replace=False, random_state=random_state))
        if n1 > 0:
            samples.append(grp_1.sample(n=n1, replace=False, random_state=random_state))
    return pd.concat(samples).reset_index(drop=True)
sample = stratified_sample(merged_df, per_stratum=15, ratio_0_1=(9,2))
sample.to_csv("Stratified_Samples.csv", index=False)

In [ ]:
# Extract 101-200 Organizations
import pandas as pd

df = pd.read_csv("Stratified_Samples.csv", skiprows=range(1, 101), nrows=100)
df.to_csv("101-200.csv", index=False)

In [10]:
# Extract Columns From Classification
import pandas as pd

orgs = pd.read_csv("101-200.csv")[["organisation_number"]]
category = pd.read_json("Charity Classification.json", encoding="utf-8-sig")
category = category[["organisation_number", "classification_type", "classification_description"]]
category = category[category["organisation_number"].isin(orgs["organisation_number"])]
types = ["What", "Who", "How"]

for t in types:
    subset = category[category["classification_type"] == t]
    subset.to_csv(f"{t}_classification.csv", index=False)
    print(f"{t}_classification.csv finished. Rows:", len(subset))

What_classification.csv finished. Rows: 264
Who_classification.csv finished. Rows: 217
How_classification.csv finished. Rows: 232


In [6]:
import pandas as pd

orgs = pd.read_excel("Filled 101-200.xlsx")[["organisation_number"]]
geo = pd.read_json("Charity Area of Operation.json", encoding="utf-8-sig")
geo = geo[["organisation_number", "geographic_area_type", "geographic_area_description"]]
result = geo[geo["organisation_number"].isin(orgs["organisation_number"])]
result.to_csv("Geo 101-200.csv", index=False)
print("Finished. Rows:", len(result))

Finished. Rows: 275


In [11]:
# Extract Columns From Partb
import pandas as pd

orgs = pd.read_excel("Filled 101-200.xlsx")[["organisation_number"]]
partb = pd.read_json("Charity Partb.json", encoding="utf-8-sig")
cols = ["organisation_number", "fin_period_end_date", "income_donations_and_legacies", "income_other_trading_activities", "income_charitable_activities", "income_investments", "income_other", "income_total_income_and_endowments", "income_endowments"]
partb = partb[cols]
partb_filtered = partb[partb["organisation_number"].isin(orgs["organisation_number"])].copy()

if "fin_period_end_date" in partb.columns:
    partb_filtered["fin_period_end_date"] = pd.to_datetime(partb_filtered["fin_period_end_date"], errors="coerce")
    partb_filtered = (partb_filtered.sort_values("fin_period_end_date", ascending=False).drop_duplicates(subset="organisation_number"))

result = pd.merge(orgs, partb_filtered, on="organisation_number", how="left")
result.to_csv("Funding Structure & Dependency.csv", index=False)
print("Finished. Rows:", len(result))

Finished. Rows: 100


In [7]:
# Extract Columns From Published Report
import pandas as pd

orgs = pd.read_excel("Filled 101-200.xlsx")[["organisation_number"]]
reports = pd.read_json("Charity Published Report.json", encoding="utf-8-sig")
reports = reports[["organisation_number", "report_name"]]
reports = reports[reports["organisation_number"].isin(orgs["organisation_number"])]
result = pd.merge(orgs, reports, on="organisation_number", how="left")
result.to_csv("Regulatory Compliance.csv", index=False)
print("Finished. Rows:", len(result))

Finished. Rows: 100


In [4]:
# Extract Columns From Partb
import pandas as pd

orgs = pd.read_excel("Filled 101-200.xlsx")[["organisation_number"]]
employee = pd.read_json("Charity Partb.json", encoding="utf-8-sig")
employee = employee[["fin_period_end_date", "organisation_number", "count_employees"]]
employee["fin_period_end_date"] = pd.to_datetime(employee["fin_period_end_date"], errors="coerce")
employee["fin_period_end_date"] = employee["fin_period_end_date"].dt.date
employee = employee[employee["organisation_number"].isin(orgs["organisation_number"])]
result = pd.merge(orgs, employee, on="organisation_number", how="left")
result.to_csv("Organization Size.csv", index=False)
print("Finished. Rows:", len(result))

Finished. Rows: 100


In [ ]:
# Calculate organisational_age
import pandas as pd

df = pd.read_excel("Filled 101-200.xlsx")
df["date_of_registration"] = pd.to_datetime(df["date_of_registration"], errors="coerce")
today = pd.Timestamp.today()
df["organisational_age"] = today.year - df["date_of_registration"].dt.year
df.loc[(today.month < df["date_of_registration"].dt.month) | ((today.month == df["date_of_registration"].dt.month) & (today.day < df["date_of_registration"].dt.day)), "organisational_age"] -= 1
df.to_csv("Filled 101-200.csv", index=False)
print("Finished updating organisational_age")

In [ ]:
# Calculate financial_ratio
import pandas as pd

df = pd.read_excel("Filled 101-200.xlsx")
df["latest_income"] = pd.to_numeric(df["latest_income"], errors="coerce")
df["latest_expenditure"] = pd.to_numeric(df["latest_expenditure"], errors="coerce")
df["operating_margin"] = (df["latest_income"] - df["latest_expenditure"]) / df["latest_income"]
df.loc[df["latest_income"] == 0, "operating_margin"] = None
df.to_csv("Filled 101-200.csv", index=False)
print("Finished calculating financial_ratio")

In [13]:
# Revenue Concentration Index Calculation
import pandas as pd
import numpy as np

df = pd.read_excel("Filled 101-200.xlsx")
income_cols = ["income_donations_and_legacies", "income_other_trading_activities", "income_charitable_activities", "income_investments", "income_other", "income_endowments"]
for col in income_cols + ["income_total_income_and_endowments"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

def compute_hhi(row):
    total = row["income_total_income_and_endowments"]
    if pd.isna(total) or total == 0:
        return np.nan
    shares = row[income_cols] / total
    return (shares**2).sum()

df["revenue_concentration_index"] = df.apply(compute_hhi, axis=1)
df.to_csv("Filled 101-200.csv", index=False)
print("Finished calculating revenue_concentration_index")

Finished calculating revenue_concentration_index


In [4]:
import pandas as pd

classification = pd.read_json("Charity Classification.json", encoding="utf-8-sig")
classification = classification[["classification_type", "classification_description"]].dropna(subset=["classification_type", "classification_description"])
classification = classification.assign(classification_type=classification["classification_type"].str.split(r'[;,/]').explode())
classification["classification_type"] = classification["classification_type"].str.strip()
classification["classification_description"] = classification["classification_description"].str.strip()
classification = classification.drop_duplicates()
category_counts = classification.groupby("classification_type")["classification_description"].nunique().reset_index()
category_counts = category_counts.rename(columns={"classification_description": "num_categories"})
print(category_counts)

  classification_type  num_categories
0                 How              10
1                What              17
2                 Who               7


In [1]:
import pandas as pd

classification = pd.read_json("Charity Classification.json", encoding="utf-8-sig")
classification = classification[["classification_type", "classification_description"]].dropna(subset=["classification_type", "classification_description"])
classification = classification.assign(classification_type=classification["classification_type"].str.split(r'[;,/]').apply(lambda x: [i.strip() for i in x]))
classification = classification.explode("classification_type")
classification["classification_type"] = classification["classification_type"].str.strip()
classification["classification_description"] = classification["classification_description"].str.strip()
classification = classification.drop_duplicates()

for ctype, group in classification.groupby("classification_type"):
    categories = group["classification_description"].unique()
    print(f"\n{'='*50}")
    print(f"Type: {ctype}  ({len(categories)} categories)")
    print(f"{'='*50}")
    for cat in sorted(categories):
        print(f"  - {cat}")


Type: How  (10 categories)
  - Acts As An Umbrella Or Resource Body
  - Makes Grants To Individuals
  - Makes Grants To Organisations
  - Other Charitable Activities
  - Provides Advocacy/advice/information
  - Provides Buildings/facilities/open Space
  - Provides Human Resources
  - Provides Other Finance
  - Provides Services
  - Sponsors Or Undertakes Research

Type: What  (17 categories)
  - Accommodation/housing
  - Amateur Sport
  - Animals
  - Armed Forces/emergency Service Efficiency
  - Arts/culture/heritage/science
  - Disability
  - Economic/community Development/employment
  - Education/training
  - Environment/conservation/heritage
  - General Charitable Purposes
  - Human Rights/religious Or Racial Harmony/equality Or Diversity
  - Other Charitable Purposes
  - Overseas Aid/famine Relief
  - Recreation
  - Religious Activities
  - The Advancement Of Health Or Saving Of Lives
  - The Prevention Or Relief Of Poverty

Type: Who  (7 categories)
  - Children/young People
  - 

In [18]:
import pandas as pd

df = pd.read_json("Charity Classification.json", encoding="utf-8-sig")
df = df[df["classification_type"].isin(["What", "Who", "How"])]
total = df["organisation_number"].nunique()

freq = (
    df.groupby(["classification_type", "classification_description"])
    ["organisation_number"].nunique()
    .reset_index()
    .rename(columns={"organisation_number": "count"})
)
freq["percentage"] = (freq["count"] / total * 100).round(1)
freq = freq.sort_values(["classification_type", "percentage"], ascending=[True, False])

for ctype, group in freq.groupby("classification_type"):
    print(f"\n{'='*55}")
    print(f"  {ctype}  (Total Charity Number: {total})")
    print(f"{'='*55}")
    print(f"{'Choice':<45} {'count':>6} {'%':>6}")
    print("-"*55)
    for _, row in group.iterrows():
        print(f"{row['classification_description']:<45} {row['count']:>6} {row['percentage']:>5}%")


  How  (Total Charity Number: 267272)
Choice                                         count      %
-------------------------------------------------------
Provides Services                             109584  41.0%
Provides Buildings/facilities/open Space       74591  27.9%
Makes Grants To Organisations                  73931  27.7%
Provides Advocacy/advice/information           71847  26.9%
Makes Grants To Individuals                    53710  20.1%
Provides Human Resources                       48701  18.2%
Other Charitable Activities                    36855  13.8%
Acts As An Umbrella Or Resource Body           23159   8.7%
Sponsors Or Undertakes Research                20046   7.5%
Provides Other Finance                         16426   6.1%

  What  (Total Charity Number: 267272)
Choice                                         count      %
-------------------------------------------------------
Education/training                            134131  50.2%
General Charitable Purposes  

In [43]:
# Apriori Interpretation
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.preprocessing import TransactionEncoder

df = pd.read_json("Charity Classification.json", encoding="utf-8-sig")
df = df[df["classification_type"].isin(["What", "Who", "How"])]
df["item"] = df["classification_type"] + "_" + df["classification_description"]
df["item"] = df["item"].astype(str)
transactions = (df.groupby("organisation_number")["item"].apply(lambda x: list(set(x))).tolist())

te = TransactionEncoder()
te_array = te.fit_transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.05, use_colnames=True, max_len=3)
frequent_itemsets["length"] = frequent_itemsets["itemsets"].apply(len)
three_combos = frequent_itemsets[
    (frequent_itemsets["length"] == 3) &
    (frequent_itemsets["itemsets"].apply(
        lambda x:
        any("What_" in i for i in x) and
        any("Who_" in i for i in x) and
        any("How_" in i for i in x)
    ))
].sort_values("support", ascending=False)
print(f"Number of frequent What+Who+How combinations: {len(three_combos)}")
print(f"\nTop 20 combinations: ")
print(three_combos.head(20).to_string())

three_combos_export = three_combos.copy()
three_combos_export["itemsets"] = three_combos_export["itemsets"].apply(lambda x: ", ".join(sorted(x)))
three_combos_export.to_excel("Frequent Combinations.xlsx", index=False)
print("Saved to Frequent Combinations.xlsx")

Number of frequent What+Who+How combinations: 61

Top 20 combinations: 
      support                                                                                                          itemsets  length
323  0.177669                                       (How_Provides Services, Who_Children/young People, What_Education/training)       3
327  0.119354                                  (Who_The General Public/mankind, How_Provides Services, What_Education/training)       3
254  0.111987                    (Who_Children/young People, What_Education/training, How_Provides Advocacy/advice/information)       3
284  0.110663                (How_Provides Buildings/facilities/open Space, Who_Children/young People, What_Education/training)       3
258  0.105350               (Who_The General Public/mankind, How_Provides Advocacy/advice/information, What_Education/training)       3
214  0.104882                           (How_Makes Grants To Organisations, Who_Children/young People, What_Educ

In [6]:
# K-means
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import silhouette_score
import ast

df = pd.read_excel('Frequent Combinations.xlsx')
df["itemsets_list"] = df["itemsets"].fillna("").apply(lambda x: [i.strip().lower() for i in x.split(",") if i.strip()])
mlb = MultiLabelBinarizer()
X = mlb.fit_transform(df["itemsets_list"])

silhouette_scores = {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    score = silhouette_score(X, labels)
    silhouette_scores[k] = score
    print(f"K={k}: silhouette score = {score:.3f}")

candidate_k = {k: v for k, v in silhouette_scores.items() if 6 <= k <= 10}
best_k = max(candidate_k, key=candidate_k.get)
print("Chosen K: ", best_k)
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["apriori_cluster"] = km_final.fit_predict(X)
df["itemset_set"] = df["itemsets_list"].apply(lambda x: frozenset(x))
pattern_map = sorted(
    zip(df["itemset_set"], df["apriori_cluster"], df["support"]),
    key=lambda x: len(x[0]),
    reverse=True
)

def assign_cluster(items):
    best_cluster = -1
    best_coverage = 0
    best_support = -1
    for pattern, cluster, sup in pattern_map:
        inter = len(pattern & items)
        if len(pattern) == 0:
            continue
        coverage = inter / len(pattern)
        if coverage > best_coverage:
            best_coverage = coverage
            best_support = sup
            best_cluster = cluster
        elif coverage == best_coverage and sup > best_support:
            best_support = sup
            best_cluster = cluster
    return best_cluster

org_df = pd.read_excel("300 Samples Labelled.xlsx")
def safe_set(x):
    if isinstance(x, set):
        return x
    if isinstance(x, str):
        try:
            return set(ast.literal_eval(x))
        except:
            return set()
    return set()

org_df["item"] = org_df.apply(
    lambda row: (
        {"what_" + str(i).strip().lower() for i in safe_set(row["what_set"])} |
        {"who_" + str(i).strip().lower() for i in safe_set(row["who_set"])} |
        {"how_" + str(i).strip().lower() for i in safe_set(row["how_set"])}
    ),
    axis=1
)

org_items = org_df[["organisation_number", "item"]].copy()
org_items["apriori_profile"] = org_items["item"].apply(assign_cluster)
df_main = pd.read_excel("300 Samples Labelled.xlsx")
df_main = df_main.drop(columns=["apriori_profile"], errors="ignore")
df_main = df_main.merge(
    org_items[["organisation_number", "apriori_profile"]],
    on="organisation_number",
    how="left"
)

df_main = df_main.drop(columns=["apriori_profile_x", "apriori_profile_y"], errors="ignore")
df_main.to_excel("300 Samples Labelled.xlsx", index=False)
print(list(org_items["item"].head(5)))
print("Updated 300 Samples Labelled.xlsx with apriori_profile")

K=2: silhouette score = 0.122
K=3: silhouette score = 0.141
K=4: silhouette score = 0.134
K=5: silhouette score = 0.123
K=6: silhouette score = 0.110
K=7: silhouette score = 0.105
K=8: silhouette score = 0.131
K=9: silhouette score = 0.116
K=10: silhouette score = 0.119
Chosen K:  8
[{'what_education/training', 'what_recreation', 'how_direct service provider', 'what_general charitable purposes', 'what_disability', 'how_advocacy or campaigning', 'who_people with disabilities'}, {'what_the prevention or relief of poverty', 'what_education/training', 'how_grant-maker or funder', 'who_children/young people', 'how_direct service provider', 'what_human rights/religious or racial harmony/equality or diversity'}, {'what_education/training', 'how_direct service provider', 'who_children/young people', 'how_advocacy or campaigning'}, {'what_the prevention or relief of poverty', 'how_grant-maker or funder', 'what_general charitable purposes', 'what_other charitable purposes', 'who_the general publ

In [29]:
# Classification Encoding
import pandas as pd

path = "300 Samples.xlsx"
main = pd.read_excel(path, sheet_name="main")
thematic_raw = pd.read_excel(path, sheet_name="thematic")
beneficiary_raw = pd.read_excel(path, sheet_name="beneficiary")
role_raw = pd.read_excel(path, sheet_name="role")

role_mapping = {
    "Makes Grants To Organisations": "Grant-maker or Funder",
    "Provides Other Finance": "Grant-maker or Funder",
    "Makes Grants To Individuals": "Grant-maker or Funder",
    "Acts As An Umbrella Or Resource Body": "Intermediary or Capacity Builder",
    "Sponsors Or Undertakes Research": "Advocacy or Campaigning",
    "Provides Advocacy/advice/information": "Advocacy or Campaigning",
    "Provides Services": "Direct Service Provider",
    "Provides Human Resources": "Intermediary or Capacity Builder",
    "Provides Buildings/facilities/open Space": "Direct Service Provider",
    "Other Charitable Activities": "Direct Service Provider"
}

main["organisation_number"] = main["organisation_number"].astype(str)
thematic_raw["organisation_number"] = thematic_raw["organisation_number"].astype(str)
beneficiary_raw["organisation_number"] = beneficiary_raw["organisation_number"].astype(str)
role_raw["organisation_number"] = role_raw["organisation_number"].astype(str)

role_raw["role_grouped"] = (role_raw["organisational_role_type"].map(role_mapping).fillna("Other"))
what_grouped = thematic_raw.groupby("organisation_number")["thematic_primary_category"].apply(lambda x: set(x.dropna()))
who_grouped = beneficiary_raw.groupby("organisation_number")["beneficiary_primary_group"].apply(lambda x: set(x.dropna()))
how_grouped = role_raw.groupby("organisation_number")["role_grouped"].apply(lambda x: set(x.dropna()))

result = main.copy()
result["organisation_number"] = result["organisation_number"].astype(str)
result = result.drop(columns=["thematic_primary_category", "beneficiary_primary_group", "organisational_role_type"], errors="ignore")
result = result.merge(what_grouped.rename("thematic_primary_category"), on="organisation_number", how="left")
result = result.merge(who_grouped.rename("beneficiary_primary_group"), on="organisation_number", how="left")
result = result.merge(how_grouped.rename("organisational_role_type"), on="organisation_number", how="left")
result["thematic_primary_category"] = result["thematic_primary_category"].apply(lambda x: x if isinstance(x, set) else set())
result["beneficiary_primary_group"] = result["beneficiary_primary_group"].apply(lambda x: x if isinstance(x, set) else set())
result["organisational_role_type"]  = result["organisational_role_type"].apply(lambda x: x if isinstance(x, set) else set())

print("\n=== Final table shape ===", result.shape)
print(result.head())
result.to_excel("300 Samples Labelled.xlsx", index=False)
print("\nSaved to 300 Samples Labelled.xlsx")


=== Final table shape === (300, 13)
  organisation_number geographic_scope_category  revenue_concentration_index  \
0             4026952             International                          NaN   
1             5043843             International                          NaN   
2             5160937             International                          NaN   
3             5219192             International                          NaN   
4             5036731             International                          NaN   

   organisational_age  latest_income  number_of_employee  \
0                  20          10550                 NaN   
1                  12          11866                 NaN   
2                   6           7177                 NaN   
3                   3           3400                 NaN   
4                  12          15592                 NaN   

   reporting_noncompliance  regulatory_action  operating_margin  \
0                        0                  0         

In [34]:
# Other Encoding
import pandas as pd
import numpy as np

df = pd.read_excel('300 Samples Labelled.xlsx')
def income_size_score(income):
    if pd.isna(income):
        return np.nan
    elif income < 100000:
        return 1
    elif income < 500000:
        return 2
    elif income < 1000000:
        return 3
    elif income < 10000000:
        return 4
    else:
        return 5
df["income_size_score"] = df["latest_income"].apply(income_size_score)

def age_score(age):
    if pd.isna(age):
        return np.nan
    elif age < 5:
        return 1
    elif age < 15:
        return 2
    elif age < 30:
        return 3
    elif age < 45:
        return 4
    elif age < 60:
        return 5
    else:
        return 6
df["age_score"] = df["organisational_age"].apply(age_score)

df["maturity"] = df["age_score"] + df["income_size_score"]
df["beneficiary_primary_group"] = df["beneficiary_primary_group"].fillna("The General Public/mankind")
df["operating_margin"] = df["operating_margin"].fillna(df["operating_margin"].median())
df.to_excel("300 Samples Labelled.xlsx", index=False)
print("Saved! ")

Saved! 


In [35]:
# Composite Variable Creation
import pandas as pd
from scipy.stats import zscore

df = pd.read_excel("300 Samples Labelled.xlsx")
df["maturity_index"] = zscore(df["maturity"], nan_policy="omit")
df["compliance_risk_score"] = df["reporting_noncompliance"] + df["regulatory_action"]
df["compliance_risk_label"] = df["compliance_risk_score"].map({
    0: "Low",
    1: "Medium",
    2: "High"
})

df = df.drop(columns=["age_score", "income_size_score", "maturity", "big org or not"], errors="ignore")
df = df[[
    "organisation_number",
    "geographic_scope_category",
    "thematic_primary_category",
    "beneficiary_primary_group",
    "organisational_role_type",
    "revenue_concentration_index",
    "organisational_age",
    "latest_income",
    "reporting_noncompliance",
    "regulatory_action",
    "operating_margin",
    "maturity_index",
    "compliance_risk_score",
    "number_of_employee",
    "compliance_risk_label",
]]
print("=== maturity_index ===")
print(df["maturity_index"].describe())
print("\n=== compliance_risk_score ===")
print(df["compliance_risk_label"].value_counts())
df.to_excel("300 Samples Labelled.xlsx", index=False)
print("\nSaved! ")

=== maturity_index ===
count    3.000000e+02
mean     3.552714e-17
std      1.001671e+00
min     -1.335760e+00
25%     -8.049946e-01
50%     -2.742289e-01
75%      7.873024e-01
max      3.441131e+00
Name: maturity_index, dtype: float64

=== compliance_risk_score ===
compliance_risk_label
Low       284
Medium     16
Name: count, dtype: int64

Saved! 
